# Download IMPACT World+ methods in your project

In [ ]:
%pip install brightway2

In [ ]:
import bw2data as bd
import bw2io as bi

In [ ]:
# Set up your Brightway project
ecoinvent_version = '3.10.1'
bd.projects.set_current(f'ecoinvent{ecoinvent_version}') # put the name of your brightway project here
regionalized = False
co2_uptake = True

## Expert version (midpoints + endpoints)

In [ ]:
if co2_uptake:
    bi.BW2Package.import_file("Data/impact_world_plus_21-incl-CO2-uptake_brightway2_expert_version_ei310.909dceb653a61d935d46530a1856f115.bw2package")
else:
    if regionalized:
        bi.BW2Package.import_file("Data/impact_world_plus_21_regionalized-for-ecoinvent-v310.0fffd5e3daa5f4cf11ef83e49c375827.bw2package")
    else:
        bi.BW2Package.import_file("Data/impact_world_plus_21_brightway2_expert_version_ei310.5535d12bedce3770ffef004e84229fd1.bw2package")

## Footprint version

In [ ]:
if not regionalized:
    bi.BW2Package.import_file("Data/impact_world_plus_21_brightway2_footprint_version_ei310.a7763e0d1b0d9263f49a7021cab8ef03.bw2package") 

## If your biosphere database is not named 'biosphere3'

In [ ]:
biosphere_db_name = f'ecoinvent-{ecoinvent_version}-biosphere'

In [ ]:
iw_methods = [i for i in bd.methods if i[0] in [
    'IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10',
    'IMPACT World+ Midpoint 2.1_regionalized for ecoinvent v3.10',
    'IMPACT World+ Damage 2.1 for ecoinvent v3.10',
    'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10',
    'IMPACT World+ Footprint 2.1 for ecoinvent v3.10',
    'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
]]

# ef_methods = [i for i in bd.methods if i[0] == 'EF v3.1 regionalized']

for method in iw_methods:
    method = bd.Method(method)
    cf_list = method.load()
    new_cf_list = []
    for cf in cf_list:
        if cf[0][0] == 'biosphere3':
            new_cf = ((biosphere_db_name, cf[0][1]), cf[1])  # replace 'biosphere3' with the name of your biosphere database
            new_cf_list.append(new_cf)
        else:
            new_cf_list.append(cf)
    method.write(new_cf_list)  # overwrite the existing method

## Remove categories other than climate change and marine acidification from in co2_uptake version

In [ ]:
iw_methods_co2_uptake = [i for i in bd.methods if i[0] in [
    'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
]]

for method in iw_methods_co2_uptake:
    method = bd.Method(method)
    if 'Climate change' not in method.name[-1] and 'Marine acidification' not in method.name[-1]:
        method.deregister()

## Add the total climate change category

In [ ]:
iw_methods_co2_uptake_damage = [i for i in bd.methods if i[0] == 'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)' and 'Climate change' in i[-1]]
eq_ccst_cf_list = []
hh_ccst_cf_list = []
eq_cclt_cf_list = []
hh_cclt_cf_list = []
for method in iw_methods_co2_uptake_damage:
    method = bd.Method(method)
    if 'Climate change, ecosystem quality, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            eq_ccst_cf_list.append(cf)
    elif 'Climate change, ecosystem quality, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            eq_cclt_cf_list.append(cf)
    elif 'Climate change, human health, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            hh_ccst_cf_list.append(cf)
    elif 'Climate change, human health, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            hh_cclt_cf_list.append(cf)

In [ ]:
iw_methods_co2_uptake_midpoint = [i for i in bd.methods if i[0] == 'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)' and 'Climate change' in i[-1]]
ccst_cf_list = []
cclt_cf_list = []
for method in iw_methods_co2_uptake_midpoint:
    method = bd.Method(method)
    if 'Climate change, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            ccst_cf_list.append(cf)
    elif 'Climate change, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            cclt_cf_list.append(cf)

In [ ]:
eq_ccst_method = bd.Method(('IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Ecosystem quality', 'Climate change, ecosystem quality, short term, total'))
eq_cclt_method = bd.Method(('IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Ecosystem quality', 'Climate change, ecosystem quality, long term, total'))
hh_ccst_method = bd.Method(('IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Human health', 'Climate change, human health, short term, total'))
hh_cclt_method = bd.Method(('IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Human health', 'Climate change, human health, long term, total'))
ccst_method = bd.Method(('IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'))
cclt_method = bd.Method(('IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)', 'Midpoint', 'Climate change, long term, total'))

In [ ]:
eq_ccst_method.register(**{'unit': 'PDF.m2.yr'})
eq_cclt_method.register(**{'unit': 'PDF.m2.yr'})
hh_ccst_method.register(**{'unit': 'DALY'})
hh_cclt_method.register(**{'unit': 'DALY'})
ccst_method.register(**{'unit': 'kg CO2-eq (short)'})
cclt_method.register(**{'unit': 'kg CO2-eq (long)'})

In [ ]:
eq_ccst_method.write(eq_ccst_cf_list)
eq_cclt_method.write(eq_cclt_cf_list)
hh_ccst_method.write(hh_ccst_cf_list)
hh_cclt_method.write(hh_cclt_cf_list)
ccst_method.write(ccst_cf_list)
cclt_method.write(cclt_cf_list)